# Review Epithelial He


## Setup


In [ ]:
from pathlib import Path as _Path
import os as _os
import sys as _sys
_candidates = (_Path.cwd(), *_Path.cwd().parents,
               _Path(globals().get("_code_root", _Path.cwd())))
_code_root = next(p for p in _candidates
                  if (p / 'xenium_pipeline/notebook_setup.py').is_file())
if str(_code_root) not in _sys.path:
    _sys.path.insert(0, str(_code_root))
from xenium_pipeline.notebook_setup import setup_notebook
_code_root, _project_root = setup_notebook()


In [ ]:
from pathlib import Path
import importlib.util
import importlib
import json
import sys
import pandas as pd
from IPython.display import display, Markdown, Image, FileLink

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (_code_root / 'xenium_pipeline').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook inside the Xenium project.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from xenium_pipeline import epithelial_morphology as morphology
importlib.reload(morphology)

from xenium_pipeline.storage import epithelial_roots
CHECKPOINT_ROOTS = epithelial_roots(ROOT)
OUTPUT_DIR = ROOT / 'epithelial_morphology_review'
REFRESH_INPUT_DISCOVERY = False
INSPECT_INPUT_ARCHIVES = False
CLUSTER_KEY = 'epithelial_r0_7'
UNRESOLVED_ANNOTATIONS = ROOT / 'epithelial_cluster_annotation' / 'all_cluster_annotations.csv'
CELL_REVIEW_DIR = OUTPUT_DIR / 'manual_cell_review'
CELL_EXAMPLES_PER_CLUSTER = 28
CELL_CROP_SIZE_PX = 384
CELL_REVIEW_SEED = 0

# Explicit paths win over discovery. Paths may be absolute or relative to ROOT.
# Set alignment_reviewed=True only after checking this sample's H&E overlay.
INPUT_OVERRIDES = {}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Locate Saved Clusters and H&E


In [ ]:
inventory_path = OUTPUT_DIR / 'input_inventory.csv'
inventory = morphology.discover_inputs(
    ROOT, CHECKPOINT_ROOTS, INPUT_OVERRIDES, cache_path=inventory_path,
    refresh=REFRESH_INPUT_DISCOVERY, inspect_archives=INSPECT_INPUT_ARCHIVES,
)
inventory.to_csv(inventory_path, index=False)
input_summary = inventory.groupby('dataset', sort=False).agg(
    n_samples=('sample', 'size'),
    n_basic_ready=('basic_ready', 'sum'),
    n_mask_files=('mask_file_present', 'sum'),
    n_boundary_files=('boundaries', lambda paths: sum(bool(p) and Path(str(p)).is_file() for p in paths)),
    n_he_file_pairs=('he_files_present', 'sum'),
).reset_index()
display(input_summary)
print('Full paths, missing inputs, ambiguous candidates, and archive members:')
display(FileLink(str(inventory_path)))
if not inventory.basic_ready.any():
    print('No source cell tables are available here. Copy this notebook and helper to the server; no clustering rerun is needed.')

## Export 28-Cell Galleries


In [ ]:
from xenium_pipeline import epithelial_image_review as image_review
importlib.reload(image_review)
if not UNRESOLVED_ANNOTATIONS.is_file():
    raise FileNotFoundError(f'Copy the saved cluster proposals here: {UNRESOLVED_ANNOTATIONS}')
image_review_targets = image_review.load_review_targets(UNRESOLVED_ANNOTATIONS, CLUSTER_KEY)
cell_gallery_index = image_review.export_cell_galleries(
    inventory, image_review_targets, CELL_REVIEW_DIR,
    count=CELL_EXAMPLES_PER_CLUSTER, crop_size_px=CELL_CROP_SIZE_PX, seed=CELL_REVIEW_SEED,
)
print(cell_gallery_index['status'].value_counts().to_string())
display(FileLink(CELL_REVIEW_DIR / 'cell_gallery_index.csv'))

## Save Accepted Labels


In [ ]:
from xenium_pipeline import epithelial_image_review as image_review
importlib.reload(image_review)
if not UNRESOLVED_ANNOTATIONS.is_file():
    raise FileNotFoundError(f'Copy the saved cluster proposals here: {UNRESOLVED_ANNOTATIONS}')
image_review_targets = image_review.load_review_targets(UNRESOLVED_ANNOTATIONS, CLUSTER_KEY)
print(f'Unresolved epithelial clusters selected: {len(image_review_targets)} (clusters in the input table)')
cell_image_reviewer = image_review.launch_image_reviewer(
    inventory, image_review_targets, CELL_REVIEW_DIR,
    count=CELL_EXAMPLES_PER_CLUSTER, crop_size_px=CELL_CROP_SIZE_PX, seed=CELL_REVIEW_SEED,
)